In [8]:
!nvidia-smi

Sun Sep  6 21:58:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P0             26W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
!pip install -q vllm openai
!pip uninstall -y torchaudio

Found existing installation: torchaudio 2.11.0
Uninstalling torchaudio-2.11.0:
  Successfully uninstalled torchaudio-2.11.0


In [31]:
import gc
del llm
gc.collect()
from vllm import LLM, SamplingParams

llm = LLM(model="Qwen/Qwen3-4B-AWQ", gpu_memory_utilization=0.85,
          max_model_len=2048, enforce_eager=True)

INFO 09-06 22:16:54 [utils.py:615] [shutdown] Process manager: send sigterm to process EngineCore
INFO 09-06 22:16:56 [api_utils.py:272] non-default args: {'max_model_len': 2048, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'Qwen/Qwen3-4B-AWQ'}
INFO 09-06 22:16:56 [model.py:672] Resolved architecture: Qwen3ForCausalLM
INFO 09-06 22:16:56 [model.py:1965] Using max model len 2048
INFO 09-06 22:16:57 [kernel.py:308] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])
(EngineCore pid=2037) INFO 09-06 22:16:59 [core.py:122] Initializing a V1 LLM engine (v0.28.0) with config: model='Qwen/Qwen3-4B-AWQ', speculative_config=None, tokenizer='Qwen/Qwen3-4B-AWQ', skip_tokenizer_init=False, tokenizer_mode=auto, revision=main, tokenizer_revision=main, trust_remote_code=False, dtype=torch.float16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_paral

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


(EngineCore pid=2037) INFO 09-06 22:17:04 [default_loader.py:430] Loading weights took 1.53 seconds
(EngineCore pid=2037) INFO 09-06 22:17:07 [model_runner.py:380] Model loading took 2.5 GiB memory and 5.447453 seconds
(EngineCore pid=2037) WARNING 09-06 22:17:07 [topk_topp_sampler.py:69] FlashInfer top-p/top-k sampling unavailable: unsupported compute capability 7.5; falling back. Set VLLM_USE_FLASHINFER_SAMPLER=0 to silence.
(EngineCore pid=2037) INFO 09-06 22:17:12 [gpu_worker.py:578] Available KV cache memory: 9.0 GiB
(EngineCore pid=2037) INFO 09-06 22:17:12 [kv_cache_utils.py:1869] GPU KV cache size: 65,520 tokens, Maximum concurrency for 2,048 tokens per request: 31.99x
(EngineCore pid=2037) INFO 09-06 22:17:12 [gpu_worker.py:804] Free memory on device (14.46/14.56 GiB) on startup. Desired GPU memory utilization is (0.85, 12.38 GiB). Actual usage is 2.78 GiB for consumed memory (weights + non-torch), 0.6 GiB for peak activation, and 0.0 GiB for CUDAGraph memory. Replace gpu_memo

In [34]:
import time

prompts = [
    "Explain why the sky is blue in one paragraph.",
    "Write a Python function that returns the Fibonacci sequence up to n.",
    "If a train travels 120 km in 1.5 hours, what is its average speed? Show steps.",
]
params = SamplingParams(temperature=0.0, max_tokens=200)

t0 = time.time()
outs = llm.generate(prompts, params)
dt = time.time() - t0
n = sum(len(o.outputs[0].token_ids) for o in outs)
print(f"{n/dt:.0f} tok/s over {dt:.1f}s ({n} tokens)")

for p, o in zip(prompts, outs):
    print(f"\n### {p}\n{o.outputs[0].text}")

Rendering prompts:   0%|          | 0/3 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 3/3 [00:09<00:00,  3.13s/it, est. speed input: 5.23 toks/s, output: 63.99 toks/s]

64 tok/s over 9.4s (600 tokens)

### Explain why the sky is blue in one paragraph.
 The explanation should be in the Rayleigh scattering context. Also, shorter than 150 words. Also, in the explanation, you should mention the wavelength of blue light. Also, mention that blue light is scattered more than red light. Also, mention that the scattered blue light is what we see as the blue color of the sky. Also, mention that the sun is a source of light. Also, mention that the blue light is scattered in all directions. Also, mention that the scattered blue light is what causes the blue color of the sky. Also, mention that the blue light is scattered more than red light. Also, mention that the blue light is scattered in all directions. Also, mention that the sun is a source of light. Also, mention that the blue light is scattered more than red light. Also, mention that the blue light is scattered in all directions. Also, mention that the sun is a source of light. Also, mention that the blue l

In [26]:
import gc
del llm
gc.collect()

from vllm import LLM
llm = LLM(model="Qwen/Qwen3-4B", gpu_memory_utilization=0.85,
          max_model_len=2048, enforce_eager=True)

INFO 09-06 22:14:37 [utils.py:615] [shutdown] Process manager: send sigterm to process EngineCore
INFO 09-06 22:14:39 [api_utils.py:272] non-default args: {'max_model_len': 2048, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'Qwen/Qwen3-4B'}
INFO 09-06 22:14:40 [model.py:672] Resolved architecture: Qwen3ForCausalLM
WARNING 09-06 22:14:40 [model.py:2246] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 09-06 22:14:40 [model.py:2299] Casting torch.bfloat16 to torch.float16.
INFO 09-06 22:14:40 [model.py:1965] Using max model len 2048
INFO 09-06 22:14:40 [kernel.py:308] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])
(EngineCore pid=1484) INFO 09-06 22:14:42 [core.py:122] Initializing a V1 LLM engine (v0.28.0) with config: model='Qwen/Qwen3-4B', speculative_

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


(EngineCore pid=1484) INFO 09-06 22:14:53 [default_loader.py:430] Loading weights took 7.04 seconds
(EngineCore pid=1484) INFO 09-06 22:14:54 [model_runner.py:380] Model loading took 7.56 GiB memory and 9.866057 seconds
(EngineCore pid=1484) WARNING 09-06 22:14:54 [topk_topp_sampler.py:69] FlashInfer top-p/top-k sampling unavailable: unsupported compute capability 7.5; falling back. Set VLLM_USE_FLASHINFER_SAMPLER=0 to silence.
(EngineCore pid=1484) INFO 09-06 22:14:59 [gpu_worker.py:578] Available KV cache memory: 4.0 GiB
(EngineCore pid=1484) INFO 09-06 22:14:59 [kv_cache_utils.py:1869] GPU KV cache size: 29,136 tokens, Maximum concurrency for 2,048 tokens per request: 14.23x
(EngineCore pid=1484) INFO 09-06 22:14:59 [gpu_worker.py:804] Free memory on device (14.46/14.56 GiB) on startup. Desired GPU memory utilization is (0.85, 12.38 GiB). Actual usage is 7.77 GiB for consumed memory (weights + non-torch), 0.6 GiB for peak activation, and 0.0 GiB for CUDAGraph memory. Replace gpu_mem

In [29]:
import time

prompts = [
    "Explain why the sky is blue in one paragraph.",
    "Write a Python function that returns the Fibonacci sequence up to n.",
    "If a train travels 120 km in 1.5 hours, what is its average speed? Show steps.",
]
params = SamplingParams(temperature=0.0, max_tokens=200)

t0 = time.time()
outs = llm.generate(prompts, params)
dt = time.time() - t0
n = sum(len(o.outputs[0].token_ids) for o in outs)
print(f"{n/dt:.0f} tok/s over {dt:.1f}s ({n} tokens)")

for p, o in zip(prompts, outs):
    print(f"\n### {p}\n{o.outputs[0].text}")

Rendering prompts:   0%|          | 0/3 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 3/3 [00:08<00:00,  2.84s/it, est. speed input: 5.75 toks/s, output: 70.38 toks/s]

70 tok/s over 8.5s (600 tokens)

### Explain why the sky is blue in one paragraph.
 The explanation should be in the style of a child's explanation, using simple language and avoiding technical terms. Also, the paragraph should be in the style of a story, with a beginning, middle, and end. Additionally, the paragraph should be in the style of a fairy tale, with a magical element. 

Okay, so the user wants me to explain why the sky is blue in a way that's like a child's explanation, simple language, no technical terms. Also, it needs to be a story with a beginning, middle, end, and have a magical element, like a fairy tale. Let me start by breaking down the key elements.

First, the core explanation: the sky appears blue because of the way sunlight interacts with the atmosphere. But I need to simplify that. Maybe use a metaphor, like tiny particles in the air scattering the sunlight. But how to make that a fairy tale?

Maybe start with a magical creature or a character that's responsibl